In [7]:
import os
import json
import requests
import psycopg2
import psycopg2.extras
from datetime import datetime
from dotenv import load_dotenv

# ── Configuración ──────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, ANTHROPIC_API_KEY]):
    raise EnvironmentError("Faltan variables: POSTGRES_*, ANTHROPIC_API_KEY")

MODELO    = "claude-sonnet-4-20250514"
PROMPT_V  = "v1"


# ── Conexión ───────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname   = POSTGRES_DB,
        user     = POSTGRES_USER,
        password = POSTGRES_PASSWORD,
        host     = POSTGRES_HOST,
        port     = POSTGRES_PORT,
    )


# ── Leer último diagnóstico sin nota AI ───────────────────────────────────────
def leer_diagnostico_pendiente(conn) -> dict | None:
    """
    Devuelve el diagnóstico más reciente que todavía no tiene nota AI.
    Si todos ya tienen nota, devuelve None.
    """
    sql = """
        SELECT d.*
        FROM macro.macro_diagnostico d
        LEFT JOIN macro.macro_notas_ai n ON n.diagnostico_id = d.id
        WHERE n.id IS NULL
        ORDER BY d.calculado_en DESC
        LIMIT 1
    """
    with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(sql)
        row = cur.fetchone()
        return dict(row) if row else None


# ── Armar el prompt ────────────────────────────────────────────────────────────
def construir_prompt(d: dict) -> str:
    """
    Recibe el dict del diagnóstico y arma el prompt para Claude.
    Le pasamos todos los datos relevantes + el estado ya calculado.
    Le pedimos que responda SOLO en JSON — sin texto adicional.
    """
    return f"""Sos un economista senior especializado en macroeconomía de USA.
Te paso los indicadores macroeconómicos actuales y el diagnóstico ya calculado por un motor de reglas determinístico.
Tu tarea es agregar contexto cualitativo — explicar el "por qué" detrás de los números.

DIAGNÓSTICO DEL MOTOR DE REGLAS:
- Estado macro:    {d['estado_macro']}
- Score de riesgo: {d['score_riesgo']} / 100
- Confianza:       {d['confianza']}
- Verdes:  {d['n_verdes']} | Amarillos: {d['n_amarillos']} | Rojos: {d['n_rojos']}
- Regla disparada: {d['regla_disparada']}

INDICADORES CLAVE:
- Desempleo:        {d['desempleo']}%    → {d['s_desempleo']}
- Fed Funds:        {d['fed_funds']}%    → {d['s_fed']}
- IPC anual:        {d['ipc_anual']}%    → {d['s_ipc']}
- IPC core anual:   {d['core_anual']}%   → {d['s_core']}
- Curva 10Y-2Y:     {d['curva_10y2y']} pp → {d['s_curva']}
- PIB trimestral:   {d['pib_trim']}%    → {d['s_pib']}
- VIX:              {d['vix']}          → {d['s_vix']}

Respondé ÚNICAMENTE con un JSON válido, sin texto antes ni después, sin bloques de código.
El JSON debe tener exactamente esta estructura:

{{
  "resumen": "2-3 oraciones explicando qué está pasando y por qué el motor clasificó como {d['estado_macro']}",
  "riesgos": "los 2-3 riesgos principales en este momento, separados por punto y coma",
  "outlook": "visión concreta para los próximos 3-6 meses",
  "score_sentimiento": <número 0-100, donde 0=muy negativo y 100=muy positivo>,
  "score_recesion": <número 0-100, probabilidad estimada de recesión en 12 meses>,
  "score_inflacion": <número 0-100, riesgo de reaceleración inflacionaria>
}}"""


# ── Llamar a Claude API ────────────────────────────────────────────────────────
def llamar_claude(prompt: str) -> tuple[dict, int]:
    """
    Llama a la API de Claude y devuelve (json_respuesta, tokens_usados).
    Espera que Claude devuelva JSON puro — si no, lanza excepción.
    """
    headers = {
        "x-api-key":         ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type":      "application/json",
    }
    body = {
        "model":      MODELO,
        "max_tokens": 1000,
        "messages":   [{"role": "user", "content": prompt}],
    }

    r = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers=headers,
        json=body,
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()

    texto         = data["content"][0]["text"].strip()
    tokens_usados = data["usage"]["input_tokens"] + data["usage"]["output_tokens"]

    # Parsear JSON — si Claude devuelve algo que no es JSON, falla acá
    try:
        parsed = json.loads(texto)
    except json.JSONDecodeError:
        # Intento de rescate: a veces Claude agrega ```json ... ```
        texto_limpio = texto.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(texto_limpio)

    return parsed, tokens_usados


# ── Guardar nota en macro_notas_ai ─────────────────────────────────────────────
def guardar_nota(conn, diagnostico: dict, nota: dict, tokens: int):
    sql = """
        INSERT INTO macro.macro_notas_ai (
            run_id, diagnostico_id, modelo_ai, estado_macro,
            resumen, riesgos, outlook, nota_completa,
            score_sentimiento, score_recesion, score_inflacion,
            tokens_usados, prompt_version
        ) VALUES (
            %(run_id)s, %(diagnostico_id)s, %(modelo_ai)s, %(estado_macro)s,
            %(resumen)s, %(riesgos)s, %(outlook)s, %(nota_completa)s,
            %(score_sentimiento)s, %(score_recesion)s, %(score_inflacion)s,
            %(tokens_usados)s, %(prompt_version)s
        )
    """
    params = {
        "run_id":            diagnostico["run_id"],
        "diagnostico_id":    diagnostico["id"],
        "modelo_ai":         MODELO,
        "estado_macro":      diagnostico["estado_macro"],
        "resumen":           nota.get("resumen"),
        "riesgos":           nota.get("riesgos"),
        "outlook":           nota.get("outlook"),
        "nota_completa":     json.dumps(nota, ensure_ascii=False),
        "score_sentimiento": nota.get("score_sentimiento"),
        "score_recesion":    nota.get("score_recesion"),
        "score_inflacion":   nota.get("score_inflacion"),
        "tokens_usados":     tokens,
        "prompt_version":    PROMPT_V,
    }
    with conn.cursor() as cur:
        cur.execute(sql, params)
    conn.commit()


# ── Pipeline principal ─────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print(f"  MACRO AI — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"{'='*65}\n")

    conn = get_conn()

    # 1. Buscar diagnóstico pendiente
    print("  Buscando diagnóstico sin nota AI...")
    diagnostico = leer_diagnostico_pendiente(conn)

    if not diagnostico:
        print("  ✓ Todos los diagnósticos ya tienen nota AI. Nada que hacer.")
        conn.close()
        return

    print(f"  → Diagnóstico encontrado: {diagnostico['estado_macro']} "
          f"| run_id: {diagnostico['run_id']} "
          f"| score: {diagnostico['score_riesgo']}")

    # 2. Construir prompt y llamar a Claude
    print("  Llamando a Claude API...")
    prompt = construir_prompt(diagnostico)

    try:
        nota, tokens = llamar_claude(prompt)
    except Exception as e:
        print(f"  ✗ Error en Claude API: {e}")
        conn.close()
        raise

    print(f"  ✓ Respuesta recibida ({tokens} tokens)")

    # 3. Mostrar en pantalla
    print(f"\n  {'─'*55}")
    print(f"  Estado:     {diagnostico['estado_macro']}")
    print(f"  Resumen:    {nota.get('resumen', '—')}")
    print(f"  Riesgos:    {nota.get('riesgos', '—')}")
    print(f"  Outlook:    {nota.get('outlook', '—')}")
    print(f"  Sentimiento:{nota.get('score_sentimiento')} / 100")
    print(f"  Recesión:   {nota.get('score_recesion')} / 100")
    print(f"  Inflación:  {nota.get('score_inflacion')} / 100")
    print(f"  {'─'*55}\n")

    # 4. Guardar en macro_notas_ai
    print("  Guardando en macro_notas_ai...")
    guardar_nota(conn, diagnostico, nota, tokens)
    print(f"  ✓ Nota guardada (run_id: {diagnostico['run_id']})")

    conn.close()

    print(f"\n{'='*65}")
    print(f"  Pipeline AI completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")


if __name__ == "__main__":
    main()


  MACRO AI — 2026-03-31 18:48

  Buscando diagnóstico sin nota AI...
  → Diagnóstico encontrado: SLOWDOWN | run_id: 20260331_1710 | score: 42
  Llamando a Claude API...
  ✓ Respuesta recibida (883 tokens)

  ───────────────────────────────────────────────────────
  Estado:     SLOWDOWN
  Resumen:    La economía estadounidense muestra señales claras de desaceleración con un crecimiento del PIB que se ha enfriado al 0.70% trimestral, mientras el desempleo se mantiene en 4.40%, sugiriendo un mercado laboral que comienza a aflojarse. El VIX elevado en 30.61 refleja alta volatilidad e incertidumbre en los mercados, mientras que la inflación persiste ligeramente por encima del objetivo de la Fed, limitando el margen de maniobra de política monetaria.
  Riesgos:    Deterioro acelerado del mercado laboral si la desaceleración se profundiza; Persistencia inflacionaria que restrinja la capacidad de la Fed para estimular la economía; Alta volatilidad financiera que amplifique la desaceleración e